In [ ]:
import torch
import torch.nn as nn
import torch.nn.functional as F

In [ ]:
class U_Net(nn.Module):
    def __init__(self, in_channels, out_channels, features=64):
        super().__init__()
        self.in_channels = in_channels # Usually 3 for RGB channels
        self.out_channels = out_channels # Number of classes
        self.features = features

        # Encoder
        self.enc1 = self.conv_block(self.in_channels, self.features)
        self.enc2 = self.conv_block(self.features, self.features*2)
        self.enc3 = self.conv_block(self.features*2, self.features*4)
        self.enc4 = self.conv_block(self.features*4, self.features*8)

        # Bottleneck
        self.bottleneck = self.conv_block(self.features*8, self.features*16)

        # Decoder
        self.upconv1 = nn.ConvTranspose2d(self.features*16, self.features*8, kernel_size=2, stride=2)
        self.dec1 = self.conv_block(self.features*16, self.features*8) # in_channels are sum of upconv and cropped encoder

        self.upconv2 = nn.ConvTranspose2d(self.features*8, self.features*4, kernel_size=2, stride=2)
        self.dec2 = self.conv_block(self.features*8, self.features*4)

        self.upconv3 = nn.ConvTranspose2d(self.features*4, self.features*2, kernel_size=2, stride=2)
        self.dec3 = self.conv_block(self.features*4, self.features*2)

        self.upconv4 = nn.ConvTranspose2d(self.features*2, self.features, kernel_size=2, stride=2)
        self.dec4 = self.conv_block(self.features*2, self.features)

        # Output segmentation map
        self.seg_map = nn.Conv2d(self.features, out_channels, kernel_size=1)


    def conv_block(self, in_channels, out_channels):
        return nn.Sequential(
            nn.Conv2d(in_channels, out_channels, kernel_size=3),
            nn.ReLU(inplace=True),
            nn.Conv2d(out_channels, out_channels, kernel_size=3),
            nn.ReLU(inplace=True)
        )

    def center_crop(self, features, target_size):
        _, _, h, w = features.size() # Extract h and w of the encoder feature map
        diff_h = h - target_size[0]
        diff_w = w - target_size[1]

        return features[:, :, diff_h // 2 : h - diff_h // 2,
                               diff_w // 2 : w - diff_w // 2]


    def forward(self, x):
        # Encoder Path
        step1 = self.enc1(x)
        pool1 = F.max_pool2d(step1, kernel_size=2)
        step2 = self.enc2(pool1)
        pool2 = F.max_pool2d(step2, kernel_size=2)
        step3 = self.enc3(pool2)
        pool3 = F.max_pool2d(step3, kernel_size=2)
        step4 = self.enc4(pool3)
        pool4 = F.max_pool2d(step4, kernel_size=2)

        # Bottleneck
        step5 = self.bottleneck(pool4)

        # Decoder Path
        up1 = self.upconv1(step5)
        cropped_step4 = self.center_crop(step4, up1.size()[2:]) # Crop step4 (encoder feature map) to match up1 size
        step6 = torch.cat((cropped_step4, up1), dim=1) # Concatenate both features
        step6 = self.dec1(step6)

        up2 = self.upconv2(step6)
        cropped_step3 = self.center_crop(step3, up2.size()[2:])
        step7 = torch.cat((cropped_step3, up2), dim=1)
        step7 = self.dec2(step7)

        up3 = self.upconv3(step7)
        cropped_step2 = self.center_crop(step2, up3.size()[2:])
        step8 = torch.cat((cropped_step2, up3), dim=1)
        step8 = self.dec3(step8)

        up4 = self.upconv4(step8)
        cropped_step1 = self.center_crop(step1, up4.size()[2:])
        step9 = torch.cat((cropped_step1, up4), dim=1)
        step9 = self.dec4(step9)

        return self.seg_map(step9)